<a href="https://colab.research.google.com/github/faisalhamza909-blip/mental-health-clustering-case-study-/blob/main/mental_health_case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
path = "/content/drive/MyDrive/case study/mental-heath-in-tech-2016_20161114.csv"
df = pd.read_csv(path)

In [6]:
df.shape

(1433, 63)

we have 1433 rows and 63 columns


In [11]:
df.head

,Are you self-employed?,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health concerns and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",...,"If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?","If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?",What is your age?,What is your gender?,What country do you live in?,What US state or territory do you live in?,What country do you work in?,What US state or territory do you work in?,Which of the following best describes your work position?,Do you work remotely?
0,0,26-100,1.0,NaN,Not eligible for coverage / N/A,NaN,No,No,I don't know,Very easy,...,Not applicable to me,Not applicable to me,39,Male,United Kingdom,NaN,United Kingdom,NaN,Back-end Developer,Sometimes
1,0,6-25,1.0,NaN,No,Yes,Yes,Yes,Yes,Somewhat easy,...,Rarely,Sometimes,29,male,United States of America,Illinois,United States of America,Illinois,Back-end Developer|Front-end Developer,Never
2,0,6-25,1.0,NaN,No,NaN,No,No,I don't know,Neither easy nor difficult,...,Not applicable to me,Not applicable to me,38,Male,United Kingdom,NaN,United Kingdom,NaN,Back-end Developer,Always
3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Sometimes,Sometimes,43,male,United Kingdom,NaN,United Kingdom,NaN,Supervisor/Team Lead,Sometimes
4,0,6-25,0.0,1.0,Yes,Yes,No,No,No,Neither easy nor difficult,...,Sometimes,Sometimes,43,Female,United States of America,Illinois,United States of America,Illinois,Executive Leadership|Supervisor/Team Lead|Dev ...,Sometimes


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1433 entries, 0 to 1432
Data columns (total 63 columns):
 #   Column                                                                                                                                                                            Non-Null Count  Dtype  
---  ------                                                                                                                                                                            --------------  -----  
 0   Are you self-employed?                                                                                                                                                            1433 non-null   int64  
 1   How many employees does your company or organization have?                                                                                                                        1146 non-null   object 
 2   Is your employer primarily a tech company/organization?     

Now we need to calculate the percentage of missing answers for each survey to see which column may neeed to be dropped or handled specially

In [23]:
missing_values = df.isnull()
mean_of_missing_values = missing_values.mean()
missing_percentage = mean_of_missing_values * 100



In [21]:
missing_percentage = missing_percentage.sort_values(ascending = False)
missing_percentage.head(15).round(1)

,0
"If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?",90.0
"If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?",85.8
Is your primary role within your company related to tech/IT?,81.6
Do you have medical coverage (private insurance or state-provided) which includes treatment of mental health issues?,80.0
"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?",80.0
"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?",80.0
"If you have revealed a mental health issue to a coworker or employee, do you believe this has impacted you negatively?",80.0
Do you know local or online resources to seek help for a mental health disorder?,80.0
Do you believe your productivity is ever affected by a mental health issue?,80.0
"If maybe, what condition(s) do you believe you have?",77.5


The table that we get after runnign above shows that columns with very high missing values are mostly caused by the survey's conditional structure (for example, questions only shown to self-employed respondents). Since these are missing due to the survey's nature, they will be dropped or filtered later on.

In [28]:
print(f"Column with more than 50% missing values: {(missing_percentage > 50).sum()}")
print(f"Column with no missing values: {(missing_percentage == 0).sum()}")


Column with more than 50% missing values: 13
Column with no missing values: 19


# **Gender**

In [46]:
gender = df["What is your gender?"]
unique_genders = gender.unique()
# print(unique_genders)
print(f"Number of unique genders: {gender.nunique()}")
print(f"Missing values: {gender.isnull().sum()}")
gender.value_counts().head(20)


Number of unique genders: 70
Missing values: 3


,count
What is your gender?,
Male,610
male,249
Female,153
female,95
M,86
m,79
F,38
f,23
Male,11


The output shows that the gender column contains 70 different answer and 3 missing values, although many of them mean the same thing. For example, "Male","male","M" all refer to male and some even contain hidden spaces.


# **AGE**

In [47]:
age = df["What is your age?"]
age.describe()


,What is your age?
count,1433.000000
mean,34.286113
std,11.290931
min,3.000000
25%,28.000000
50%,33.000000
75%,39.000000
max,323.000000


In [53]:
unrealistic_ages = age[(age < 16) | (age>75)]
print(f"Number of unrealistic ages : {len(unrealistic_ages)}")
print(unrealistic_ages)

Number of unrealistic ages : 4
372     99
564    323
656      3
808     15
Name: What is your age?, dtype: int64


The age column has no missing values and the median age is 33 with most common age between 28 and 39. A few valus are unrealistic. They are most likely errors or typos or too young for regular employment. It is also very unlikely that somebody below the age of 16 would work in a tech company while 17 year olds could plausibly be interns or work voluntarily.